# 23. Kiwi BM25 단독 검색: GPT-5 mini와 Qwen3.8 답변 비교

**3개 사업 × 10문항, 총 30문항**에서 Kiwi BM25 단독 검색을 한 번 실행합니다. 검색 상위 20개 중 앞의 4개를 두 모델에 **똑같이** 전달하고 답변을 각각 30개 생성합니다. 질문·검색 문서·프롬프트를 고정해 생성 모델에 따른 답변 차이를 볼 수 있습니다.

- 검색은 21번의 문서 로딩·청킹(700/100)·사업 식별·Kiwi 형태소 분석·사업별 BM25 상위 20개 설정을 따릅니다. Dense, RRF, 재순위화는 사용하지 않습니다.
- OpenAI 쪽은 이전 GPT 실험에 사용한 `gpt-5-mini-2025-08-07`입니다. 20번 파일은 하이브리드 검색과 재순위화를 사용했으므로, 그때 저장한 답변을 이번 Qwen 답변과 직접 비교하면 검색 조건이 다릅니다. 이 파일에서 GPT 답변을 새로 생성합니다.
- Qwen 쪽은 [`Qwen3.8-27B`](https://huggingface.co/Qwen/Qwen3.8-27B)의 [Q3_K_S GGUF 양자화](https://huggingface.co/bartowski/Qwen3.8-27B-GGUF) 모델을 무료 Colab T4에서 llama.cpp로 실행합니다. 실제 T4 전체 실행은 아직 확인하지 않았습니다.
- Drive의 `bm25_only_comparison_30.json`이 있으면 21번의 청크와 검색 순위가 같은지 확인합니다. 없어도 두 생성 모델의 공정한 쌍별 비교는 실행할 수 있습니다.
- 답변은 한 개씩 Drive에 저장하며, 다시 실행하면 완료된 답변을 이어서 사용합니다. LangSmith는 기본적으로 실행하지 않습니다.

**준비:** Colab에서 T4 GPU를 선택하고 Drive의 `MyDrive/data`에 RFP 문서, `MyDrive/data_list.csv`에 목록, `MyDrive/rag_eval_dataset_3x10.json`에 질문 30개를 둡니다. Colab 비밀에 `OPENAI_API_KEY`를 등록합니다. GPT 호출에는 API 비용이 들고 Qwen 파일 다운로드와 빌드에는 시간이 걸립니다. 위에서 아래로 실행하세요.


In [ ]:
%pip install -q pandas pymupdf4llm langchain-hwp-hwpx-loader langchain-text-splitters langchain-community rank_bm25 kiwipiepy huggingface_hub openai langsmith


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


## 1. 문서를 21번과 같은 방식으로 읽기

HWP/HWPX 표는 마크다운으로 추출하고 CSV의 사업 정보를 붙입니다.


In [ ]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()

df = pd.read_csv(metadata_csv, encoding="utf-8-sig")
loaded_files = []
for name in sorted_file_names:
    full_path = os.path.join(folder, name)
    if not os.path.isfile(full_path):
        continue
    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp", "hwpx"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = "\n\n".join(doc.page_content for doc in loader.load())
    else:
        continue
    if not text.strip():
        raise ValueError(f"본문이 비어 있습니다: {name}")
    loaded_files.append((name, text))
if not loaded_files:
    raise ValueError(f"PDF/HWP/HWPX 문서를 찾지 못했습니다: {folder}")
print("로드한 문서:", len(loaded_files))
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]
missing_columns = set(metadata_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"data_list.csv에 없는 열: {sorted(missing_columns)}")
documents = []
unmatched_files = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        unmatched_files.append(name)
        continue
    if len(result_df[["발주 기관", "사업명"]].drop_duplicates()) != 1:
        raise ValueError(f"파일의 사업 메타데이터가 여러 개입니다: {name}")
    row = result_df.iloc[0]
    metadata = {column: "" if pd.isna(row[column]) else str(row[column])
                for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))
if not documents:
    raise ValueError("문서 파일명과 data_list.csv의 파일명을 확인하세요.")
print("메타데이터와 연결한 문서:", len(documents))
if unmatched_files:
    print("CSV와 연결되지 않아 제외한 파일:", unmatched_files)
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 2. 청킹과 사업별 Kiwi BM25

21번과 같은 청크와 형태소 분석으로 사업별 검색기를 준비합니다.


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))

from collections import defaultdict
from langchain_community.retrievers import BM25Retriever


def normalize_project_text(value):
    return "".join(
        char for char in unicodedata.normalize("NFKC", str(value)).casefold()
        if char.isalnum()
    )


def project_title_matches_query(project_key, normalized_query):
    agency, title = project_key
    normalized_title = normalize_project_text(title)
    if normalized_title in normalized_query:
        return True

    # 기관명이 사업명 앞에 있지만 질문에서는 조사와 함께 분리된 경우
    normalized_agency = normalize_project_text(agency)
    if normalized_agency and normalized_title.startswith(normalized_agency):
        short_title = normalized_title[len(normalized_agency):]
        if len(short_title) >= 8 and normalized_agency in normalized_query:
            return short_title in normalized_query
    return False


def metadata_text(value):
    return "" if pd.isna(value) else str(value).strip()


# 줄임말을 쓰려면 아래에 "줄임말": ("발주 기관", "정식 사업명")을 추가합니다.
PROJECT_ALIASES = {}

project_chunks_by_key = defaultdict(list)
for doc in chunks:
    agency = metadata_text(doc.metadata.get("발주 기관"))
    title = metadata_text(doc.metadata.get("사업명"))
    if title:
        project_chunks_by_key[(agency, title)].append(doc)

project_catalog = sorted(project_chunks_by_key)
if not project_catalog:
    raise ValueError("사업명 메타데이터가 있는 청크가 없습니다.")
print("식별 가능한 사업:", len(project_catalog))


def choose_project(matches, query):
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    normalized_query = normalize_project_text(query)
    agency_matches = [
        key for key in matches
        if key[0] and normalize_project_text(key[0]) in normalized_query
    ]
    if len(agency_matches) == 1:
        return agency_matches[0]

    labels = ", ".join(f'[{agency}] "{title}"' for agency, title in matches)
    raise ValueError(f"사업을 하나로 특정할 수 없습니다: {labels}")


def identify_project(query):
    """질문의 정식 사업명 또는 등록된 줄임말로 사업 하나를 고릅니다."""
    normalized_query = normalize_project_text(query)
    if not normalized_query:
        raise ValueError("질문이 비어 있습니다.")

    # 정식 사업명이 질문에 들어 있으면 포함 관계의 짧은 이름만 제외합니다.
    direct_matches = [
      key for key in project_catalog
      if project_title_matches_query(key, normalized_query)
    ]
    if direct_matches:
        specific_matches = [
            key for key in direct_matches
            if not any(
                normalize_project_text(key[1]) != normalize_project_text(other[1])
                and normalize_project_text(key[1]) in normalize_project_text(other[1])
                for other in direct_matches
            )
        ]
        return choose_project(specific_matches, query)

    # 정식 명칭이 없는 질문은 사용자가 등록한 줄임말만 사용합니다.
    alias_matches = [
        (alias, target) for alias, target in PROJECT_ALIASES.items()
        if normalize_project_text(alias) in normalized_query
    ]
    if alias_matches:
        longest = max(len(normalize_project_text(alias)) for alias, _ in alias_matches)
        targets = [
            target for alias, target in alias_matches
            if len(normalize_project_text(alias)) == longest
        ]
        unknown = [target for target in targets if target not in project_chunks_by_key]
        if unknown:
            raise ValueError(f"PROJECT_ALIASES의 사업이 문서 목록에 없습니다: {unknown}")
        return choose_project(targets, query)

    raise ValueError(
        "질문에서 사업명을 찾지 못했습니다. 정식 사업명을 질문에 적거나 "
        "PROJECT_ALIASES에 줄임말을 등록해 주세요."
    )


from kiwipiepy import Kiwi

kiwi = Kiwi()
PUNCTUATION_TAGS = {"SF", "SP", "SS", "SE", "SO", "SW"}


def kiwi_morphs(text):
    """문서와 질문에 같은 Kiwi 형태소 분석을 적용합니다."""
    return [
        token.form
        for token in kiwi.tokenize(text)
        if token.form.strip() and token.tag not in PUNCTUATION_TAGS
    ]



SEARCH_TOP_K = 20
CONTEXT_TOP_K = 4


def make_kiwi_index(project_key):
    # 21번과 같은 사업별 Kiwi BM25 검색기를 만듭니다.
    docs = project_chunks_by_key[project_key]
    if len(docs) < SEARCH_TOP_K:
        raise ValueError(f'사업의 청크가 {SEARCH_TOP_K}개보다 적습니다: {project_key}')
    index = BM25Retriever.from_documents(docs, preprocess_func=kiwi_morphs)
    index.k = SEARCH_TOP_K
    return index


## 3. 같은 30문항 불러오기

21번과 같이 앞의 3개 사업에서 각 10문항을 고릅니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

EVAL_PROJECT_COUNT = 3
EVAL_QUESTIONS_PER_PROJECT = 10
EXPECTED_QUESTION_COUNT = EVAL_PROJECT_COUNT * EVAL_QUESTIONS_PER_PROJECT
EVAL_LABEL = f"{EVAL_PROJECT_COUNT}x{EVAL_QUESTIONS_PER_PROJECT}"
eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset_3x10.json"),
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset_3x10.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((p for p in eval_dataset_candidates if p.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError("평가 JSON을 Google Drive 루트 또는 노트북 폴더에 두세요.")
with eval_dataset_path.open(encoding="utf-8-sig") as file:
    all_eval_records = json.load(file)
if not isinstance(all_eval_records, list) or not all_eval_records:
    raise ValueError("평가 JSON은 문항 객체들의 비어 있지 않은 목록이어야 합니다.")
if any(not isinstance(r, dict) or not r.get("project") for r in all_eval_records):
    raise ValueError("사업명(project)이 없는 평가 문항이 있습니다.")
available_projects = list(dict.fromkeys(r["project"] for r in all_eval_records))
selected_projects = available_projects[:EVAL_PROJECT_COUNT]
selected_eval_records = []
selected_count = Counter()
for record in all_eval_records:
    project = record["project"]
    if project in selected_projects and selected_count[project] < EVAL_QUESTIONS_PER_PROJECT:
        selected_eval_records.append(record)
        selected_count[project] += 1
if len(selected_projects) != EVAL_PROJECT_COUNT or any(
    selected_count[p] != EVAL_QUESTIONS_PER_PROJECT for p in selected_projects
):
    raise ValueError(f"3개 사업에서 각 10문항을 선택하지 못했습니다: {dict(selected_count)}")
evaluation_data = []
for record in selected_eval_records:
    fields = ("id", "question", "reference", "project")
    if not all(isinstance(record.get(f), str) and record[f].strip() for f in fields):
        raise ValueError(f"필수 평가 필드가 비어 있거나 문자열이 아닙니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"], "query": record["question"],
        "reference": record["reference"], "project": record["project"],
        "question_type": record.get("question_type", ""),
        "evidence_quotes": record.get("evidence_quotes", []),
    })
for field in ("id", "query"):
    if len({r[field] for r in evaluation_data}) != EXPECTED_QUESTION_COUNT:
        raise ValueError(f"문항 수가 맞지 않거나 {field}가 중복됩니다.")
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("사업별:", dict(Counter(r["project"] for r in evaluation_data)))
print("유형별:", dict(Counter(r["question_type"] for r in evaluation_data)))


## 4. Kiwi BM25 단독 검색

30문항에서 상위 20개를 검색하고, 21번 저장 결과가 있으면 순위가 같은지 확인합니다.


In [ ]:
import hashlib

# 21번과 같은 30문항의 사업별 Kiwi BM25 상위 20개를 구합니다.
project_keys = {item['id']: identify_project(item['query']) for item in evaluation_data}
indexes = {}
corpus = {}
for key in dict.fromkeys(project_keys.values()):
    docs = project_chunks_by_key[key]
    indexes[key] = make_kiwi_index(key)
    for doc in docs:
        chunk_id = str(doc.metadata['chunk_id'])
        corpus[chunk_id] = {
            'text': doc.page_content,
            'agency': key[0],
            'project': key[1],
        }

corpus_sha256 = hashlib.sha256(
    json.dumps(corpus, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()
dataset_sha256 = hashlib.sha256(
    json.dumps(evaluation_data, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()

retrievals = {}
for item in evaluation_data:
    key = project_keys[item['id']]
    found = indexes[key].invoke(item['query'])
    retrievals[item['id']] = {
        'project_key': key,
        'documents': found,
        'ranked_chunk_ids': [str(doc.metadata['chunk_id']) for doc in found],
    }

# 21번 저장 결과가 있으면 실제 검색 순위까지 대조합니다.
baseline_path = Path('/content/drive/MyDrive/bm25_only_comparison_30.json')
if baseline_path.is_file():
    baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
    if baseline.get('comparison_type') != 'bm25_only':
        raise ValueError('21번의 단독 검색 결과 파일이 아닙니다.')
    if baseline.get('corpus_sha256') != corpus_sha256:
        raise ValueError('21번 결과와 문서·청크가 다릅니다. 같은 자료로 21번을 다시 실행하세요.')
    previous = {row['id']: row for row in baseline['questions']}
    for item in evaluation_data:
        old = previous.get(item['id'])
        if old is None or old['question'] != item['query']:
            raise ValueError(f"21번 결과와 문항이 다릅니다: {item['id']}")
        if old['ranked_chunk_ids']['kiwi'] != retrievals[item['id']]['ranked_chunk_ids']:
            raise ValueError(f"21번 결과와 Kiwi 검색 순위가 다릅니다: {item['id']}")
    print('21번과 문서·청크 및 30문항의 Kiwi 검색 순위가 모두 일치합니다.')
else:
    print('21번 저장 JSON이 없어 이전 실행과의 일치 검사는 생략합니다.')
print('검색 완료:', len(retrievals), '문항 / 각 상위', SEARCH_TOP_K, '개')


## 5. GPT-5 mini 준비

기존 OpenAI 생성 모델을 같은 검색 조건에서 다시 실행합니다.


In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENAI_API_KEY') or os.environ.get('OPENAI_API_KEY')
if not api_key:
    raise ValueError('Colab 비밀에 OPENAI_API_KEY를 등록하세요.')

GPT_MODEL = 'gpt-5-mini-2025-08-07'
gpt_client = OpenAI(api_key=api_key, timeout=180, max_retries=4)
print('OpenAI 생성 모델:', GPT_MODEL)


## 6. Qwen3.8 준비

무료 Colab T4에서 GGUF 양자화 모델을 다운로드하고 로컬 서버를 실행합니다. 첫 실행은 오래 걸릴 수 있습니다.


In [ ]:
import subprocess
import time
import requests
import torch
from huggingface_hub import hf_hub_download

if not torch.cuda.is_available():
    raise RuntimeError('Colab 메뉴에서 런타임 > 런타임 유형 변경 > T4 GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))

LLAMA_DIR = Path('/content/llama.cpp')
if not LLAMA_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)],
        check=True,
    )
subprocess.run([
    'cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR / 'build'),
    '-DGGML_CUDA=ON', '-DCMAKE_CUDA_ARCHITECTURES=75',
    '-DLLAMA_BUILD_TESTS=OFF', '-DCMAKE_BUILD_TYPE=Release',
], check=True)
subprocess.run([
    'cmake', '--build', str(LLAMA_DIR / 'build'), '--config', 'Release',
    '-j', '2', '--target', 'llama-server',
], check=True)

QWEN_REPO = 'bartowski/Qwen3.8-27B-GGUF'
QWEN_FILE = 'Qwen3.8-27B-Q3_K_S.gguf'
model_path = hf_hub_download(repo_id=QWEN_REPO, filename=QWEN_FILE)

server_url = 'http://127.0.0.1:8080'
server_log = open('/content/qwen38_server.log', 'w')
server = subprocess.Popen([
    str(LLAMA_DIR / 'build/bin/llama-server'),
    '-m', str(model_path), '--host', '127.0.0.1', '--port', '8080',
    '--gpu-layers', 'auto', '--fit', 'on',
    '-c', '4096', '--parallel', '1', '--reasoning', 'off',
    '--chat-template-kwargs', '{"enable_thinking": false}',
], stdout=server_log, stderr=subprocess.STDOUT)

for _ in range(180):
    if server.poll() is not None:
        raise RuntimeError('Qwen 서버가 중단됐습니다. /content/qwen38_server.log를 확인하세요.')
    try:
        if requests.get(server_url + '/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError('Qwen 서버 시작 시간이 초과됐습니다. /content/qwen38_server.log를 확인하세요.')

qwen_client = OpenAI(base_url=server_url + '/v1', api_key='local', timeout=600)
QWEN_MODEL = qwen_client.models.list().data[0].id
print('Qwen 생성 모델:', QWEN_MODEL)


## 7. 두 모델에 같은 문서와 질문 전달

검색 상위 4개와 프롬프트를 같게 둡니다. 모델별 생성 방식과 내부 설정은 서로 다릅니다.


In [ ]:
SYSTEM_PROMPT = (
    '너는 RFP 문서 질문 답변 도우미다. 제공된 문서 내용만 근거로 한국어로 답한다. '
    '모르는 내용은 추측하지 말고 확인할 수 없다고 말한다. '
    '중요한 사실 뒤에는 근거 번호 [1]처럼 붙인다.'
)


def make_messages(question, documents):
    # 두 모델 모두 검색 결과 상위 4개의 동일한 본문과 프롬프트를 받습니다.
    context = '\n\n'.join(
        f'[{number}] 파일: {doc.metadata["파일명"]} / 청크: {doc.metadata["chunk_id"]}\n{doc.page_content}'
        for number, doc in enumerate(documents[:CONTEXT_TOP_K], 1)
    )
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'<문서>\n{context}\n</문서>\n\n<질문>\n{question}\n</질문>'},
    ]


def generate_gpt(messages):
    response = gpt_client.chat.completions.create(
        model=GPT_MODEL, messages=messages, max_completion_tokens=2048,
    )
    answer = (response.choices[0].message.content or '').strip()
    if not answer:
        raise ValueError('GPT 답변이 비어 있습니다.')
    return answer


def generate_qwen(messages):
    response = qwen_client.chat.completions.create(
        model=QWEN_MODEL, messages=messages, max_tokens=768,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}},
    )
    answer = (response.choices[0].message.content or '').strip()
    if not answer:
        raise ValueError('Qwen 답변이 비어 있습니다.')
    return answer


## 8. 30문항 답변 생성과 비교

각 문항의 GPT·Qwen 답변을 Drive에 바로 저장합니다. 재실행하면 이미 저장한 답변은 다시 만들지 않습니다. 마지막 표와 CSV에서 참고 답변과 두 생성 결과를 나란히 봅니다. 자동 채점은 하지 않습니다.


In [ ]:
OUTPUT_JSON = Path('/content/drive/MyDrive/kiwi_bm25_gpt5mini_vs_qwen38_30.json')
OUTPUT_CSV = OUTPUT_JSON.with_suffix('.csv')
settings = {
    'gpt_model': GPT_MODEL,
    'qwen_repo': QWEN_REPO,
    'qwen_file': QWEN_FILE,
    'search_top_k': SEARCH_TOP_K,
    'context_top_k': CONTEXT_TOP_K,
    'corpus_sha256': corpus_sha256,
    'dataset_sha256': dataset_sha256,
}

if OUTPUT_JSON.is_file():
    saved = json.loads(OUTPUT_JSON.read_text(encoding='utf-8'))
    if saved.get('settings') != settings:
        raise ValueError('저장된 결과의 문서·질문·모델 설정이 다릅니다. 다른 출력 파일명을 사용하세요.')
else:
    saved = {'settings': settings, 'results': []}

saved_by_id = {row['id']: row for row in saved['results']}
expected_ids = {item['id'] for item in evaluation_data}
if len(saved_by_id) != len(saved['results']) or not set(saved_by_id) <= expected_ids:
    raise ValueError('저장된 결과에 중복 또는 예상하지 않은 문항 ID가 있습니다.')


def save_progress():
    OUTPUT_JSON.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding='utf-8')


for number, item in enumerate(evaluation_data, 1):
    question_id = item['id']
    search = retrievals[question_id]
    row = saved_by_id.get(question_id)
    if row is None:
        row = {
            'id': question_id,
            'project': item['project'],
            'question': item['query'],
            'reference': item['reference'],
            'search_chunk_ids': search['ranked_chunk_ids'],
            'context_chunk_ids': search['ranked_chunk_ids'][:CONTEXT_TOP_K],
            'gpt_answer': '',
            'qwen_answer': '',
        }
        saved['results'].append(row)
        saved_by_id[question_id] = row
    messages = make_messages(item['query'], search['documents'])

    if not row['gpt_answer']:
        row['gpt_answer'] = generate_gpt(messages)
        save_progress()
    if not row['qwen_answer']:
        row['qwen_answer'] = generate_qwen(messages)
        save_progress()
    print(f'[{number}/30] {question_id}: GPT와 Qwen 답변 저장 완료')

ordered_results = [saved_by_id[item['id']] for item in evaluation_data]
assert len(ordered_results) == 30
assert all(row['gpt_answer'] and row['qwen_answer'] for row in ordered_results)

table = pd.DataFrame(ordered_results)[
    ['id', 'project', 'question', 'reference', 'gpt_answer', 'qwen_answer', 'context_chunk_ids']
]
table.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('30문항 비교 완료. 저장:', OUTPUT_JSON, '및', OUTPUT_CSV)
display(table)


## 9. LangSmith 기록 (기본 실행 안 함)

필요할 때만 `RUN_LANGSMITH`를 `True`로 바꾸세요. 저장된 결과를 기록하며 모델을 다시 호출하지 않습니다.


In [ ]:
RUN_LANGSMITH = False

if RUN_LANGSMITH:
    from langsmith import traceable

    langsmith_key = userdata.get('LANGSMITH_API_KEY')
    if not langsmith_key:
        raise ValueError('Colab 비밀에 LANGSMITH_API_KEY를 등록하세요.')
    os.environ['LANGSMITH_API_KEY'] = langsmith_key
    os.environ['LANGSMITH_TRACING'] = 'true'
    os.environ['LANGSMITH_PROJECT'] = 'rfp-kiwi-gpt-vs-qwen-30'

    @traceable(name='saved_gpt_qwen_pair')
    def log_saved_pair(result):
        return result

    for row in ordered_results:
        log_saved_pair(row)
    print('저장된 30쌍을 LangSmith에 기록했습니다. 답변은 다시 생성하지 않았습니다.')
else:
    print('LangSmith 실행 안 함')
